# Calligraphy agent: phase-1 SFT with Unsloth, served and evaluated through Together AI

This notebook is the first step of the training loop in `docs/IMPLEMENTATION.md`: supervised fine-tuning of a Qwen3-VL model to write a Chinese character stroke by stroke from its name alone (phase 1: one JSON reply, no image, no tools). The zero-shot baseline scores 0.02 against a 0.91 ceiling, so SFT comes before any RL.

What it does, in order:

1. **Data.** Clones the repo, downloads Make Me a Hanzi, and exports the SFT set with `scripts/make_sft.py`: the exact evaluation prompt plus the converter's answer, 8,692 training characters (bench characters are excluded by the frozen splits).
2. **Baseline.** Scores the untrained model on a few `viz16` characters with the training reward, and shows the drawings.
3. **Train (Unsloth).** LoRA on the language layers of `Qwen3-VL-8B-Instruct`, loss on the answer only.
4. **Evaluate.** Same reward, same characters, after training: contact sheet of reference vs model, mean reward and parse rate. Saves the adapter (and optionally a merged 16-bit model) to Drive / the Hub.
5. **Together AI (optional).** Uploads the same JSONL and launches a managed fine-tuning job, then deploys the result on a dedicated endpoint so the repo's agent harness (`scripts/run_agent.py --provider together`) can evaluate it on `val`, `test_compositional` and the bench. Together's serverless LoRA upload was discontinued, so a dedicated endpoint is the only hosted route; its fine-tuning jobs are a second, GPU-free way to train the same data.

**Runtime.** Pick an **A100 (40 GB)** or **L4** GPU (Runtime → Change runtime type). Answers are about 3k tokens (14 tokens per point at frame 1024), so one epoch over 8.7k characters at an effective batch of 16 is about 540 steps; expect 1.5–3 h on an A100. A free **T4** runs the 4-bit model for a smoke test (`MAX_STEPS = 30`) but is too slow for a full epoch.

**Secrets** (Colab → key icon in the sidebar, or environment variables): `HF_TOKEN` (optional, to push), `TOGETHER_API_KEY` (section 5), `WANDB_API_KEY` (optional), `GITHUB_TOKEN` (only if the repo is private).

## 0. Configuration

Everything you might change is here. Defaults follow the implementation doc: LoRA rank 32 on the language layers, learning rate 1e-4, action frame 1024 (frame 64 would cost 0.14 reward on the ceiling).

In [ ]:
# @title Run configuration { display-mode: "form" }
# ---- repo and data ----
REPO_URL = "https://github.com/alex-wang101/caligraphy-pt.git"  # @param {type:"string"}
REPO_BRANCH = "main"  # @param {type:"string"}
ACTION_FRAME = 1024  # @param [1024, 256, 64] {type:"raw"}
RDP_EPS = 2.0  # @param {type:"number"}   # converter point budget: 2.0 -> ~10 points/stroke, 0.5 -> ~14 (higher-fidelity labels, longer answers)
TRAIN_LIMIT = 0  # @param {type:"integer"}   # 0 = all 8,692 training characters

# ---- model ----
BASE_MODEL = "unsloth/Qwen3-VL-8B-Instruct"  # @param ["unsloth/Qwen3-VL-8B-Instruct", "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit", "unsloth/Qwen3-VL-4B-Instruct", "unsloth/Qwen3-VL-2B-Instruct"] {allow-input: true}
LOAD_IN_4BIT = False  # @param {type:"boolean"}   # True on a T4 / 16 GB card; False (bf16 LoRA) on an A100
MAX_SEQ_LEN = 4096  # @param {type:"integer"}   # prompt ~300 tokens + answer <= 3.7k (33-stroke 鱻)
LORA_R = 32  # @param {type:"integer"}
LORA_ALPHA = 32  # @param {type:"integer"}

# ---- training ----
LEARNING_RATE = 1e-4  # @param {type:"number"}
NUM_EPOCHS = 1  # @param {type:"number"}
MAX_STEPS = -1  # @param {type:"integer"}   # -1 = run NUM_EPOCHS; 30 = smoke test
PER_DEVICE_BATCH = 2  # @param {type:"integer"}
GRAD_ACCUM = 8  # @param {type:"integer"}   # effective batch 16
WARMUP_RATIO = 0.03  # @param {type:"number"}
EVAL_ROWS = 32  # @param {type:"integer"}   # val rows for the loss curve during training (0 = off)
SEED = 3407  # @param {type:"integer"}

# ---- evaluation with the reward ----
EVAL_SPLIT = "viz16"  # @param ["viz16", "val", "test_compositional"]
EVAL_LIMIT = 16  # @param {type:"integer"}
EVAL_TEMPERATURE = 0.0  # @param {type:"number"}   # 0 = greedy; the RL plan samples at 1.0
EVAL_MAX_NEW_TOKENS = 4096  # @param {type:"integer"}
BASELINE_CHARS = 4  # @param {type:"integer"}   # untrained-model check before training (0 = skip)

# ---- outputs ----
RUN_NAME = "sft-qwen3vl8b-r32-f1024"  # @param {type:"string"}
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
HF_REPO = ""  # @param {type:"string"}   # e.g. "your-name/calligraphy-sft-lora"; empty = don't push
PUSH_MERGED_16BIT = False  # @param {type:"boolean"}   # merged weights for vLLM / Together custom models (16 GB upload)
REPORT_TO = "none"  # @param ["none", "wandb"]

# ---- Together AI (section 5) ----
TOGETHER_MODEL = "Qwen/Qwen3-VL-8B-Instruct"  # @param ["Qwen/Qwen3-VL-8B-Instruct", "Qwen/Qwen3.5-9B", "Qwen/Qwen3.5-4B"] {allow-input: true}
TOGETHER_EPOCHS = 2  # @param {type:"integer"}
TOGETHER_LR = 1e-4  # @param {type:"number"}
TOGETHER_LORA_R = 32  # @param {type:"integer"}
TOGETHER_SUFFIX = "calligraphy-sft"  # @param {type:"string"}

## 1. Install and set up

The Unsloth install block is the one from Unsloth's own Qwen3-VL notebook (pinned `transformers` / `trl`). The repo's core is pure Python and only needs `shapely`, `perfect-freehand` and `pyarrow` on top of Colab's numpy and Pillow.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2
!pip install "together[cli]" shapely perfect-freehand pyarrow

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

def secret(name):
    """Colab secret if set, else environment variable, else None."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
    except Exception:
        v = None
    return v or os.environ.get(name)

for name in ("HF_TOKEN", "TOGETHER_API_KEY", "WANDB_API_KEY", "GITHUB_TOKEN"):
    v = secret(name)
    if v:
        os.environ[name] = v
    print(f"{name}: {'set' if v else 'not set'}")

# Clone the repo and put its core on the path (the package is pure Python: no build step).
REPO = Path("/content/caligraphy-pt")
if not REPO.exists():
    url = REPO_URL
    if os.environ.get("GITHUB_TOKEN") and url.startswith("https://github.com/"):  # private repo
        url = url.replace("https://", f"https://{os.environ['GITHUB_TOKEN']}@", 1)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, str(REPO)], check=True)
sys.path[:0] = [str(REPO / "src"), str(REPO / "scripts")]
os.chdir(REPO)

# Datasets: Make Me a Hanzi (33 MB) and calligraphy-bench (0.6 MB) into data/.
subprocess.run([sys.executable, "scripts/fetch_data.py"], check=True)

# Where results go.
OUT_DIR = Path("/content/out") / RUN_NAME
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = Path("/content/drive/MyDrive/calligraphy") / RUN_NAME
    except Exception as e:
        print("Drive not mounted, saving under /content/out:", e)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR)

## 2. Build the SFT dataset

`make_sft.py` writes one chat example per character: the system prompt and user turn are exactly what `run_agent.py` sends at evaluation time (`phase1_messages`), and the assistant turn is the converter's answer. The script scores the first 20 labels of each split with the reward as a sanity check (expect about 0.92).

In [ ]:
sft_dir = OUT_DIR / "sft"
cmd = [sys.executable, "scripts/make_sft.py", "--splits", "train", "val", "viz16",
       "--action-frame", str(ACTION_FRAME), "--rdp-eps", str(RDP_EPS), "--out", str(sft_dir)]
if TRAIN_LIMIT:
    cmd += ["--limit", str(TRAIN_LIMIT)]
subprocess.run(cmd, check=True)

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

train_rows = read_jsonl(sft_dir / "train.jsonl")
val_rows = read_jsonl(sft_dir / "val.jsonl")
print(len(train_rows), "train rows,", len(val_rows), "val rows")
print(train_rows[0]["messages"][1]["content"])
print(train_rows[0]["messages"][2]["content"][:300], "...")

## 3. Load the model and add LoRA

Phase 1 never shows the model an image, so the vision tower stays frozen (`finetune_vision_layers=False`); the LoRA goes on the attention and MLP projections of the language model, rank 32. The vision tower is kept in the checkpoint so the same adapter can continue into phase 2 (`capture_canvas`).

In [ ]:
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=LOAD_IN_4BIT,
    use_gradient_checkpointing="unsloth",
)
tok = getattr(tokenizer, "tokenizer", tokenizer)  # the text tokenizer inside the processor
tok.padding_side = "left"

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=False,
)
model.print_trainable_parameters()

### Convert to Unsloth's conversation format and check lengths

Unsloth's vision collator takes `content` as a list of parts; text-only rows are supported (it batches image rows and text-only rows separately). Rows longer than `MAX_SEQ_LEN` would be truncated mid-answer, so they are dropped and counted here instead.

In [ ]:
def to_parts(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

def n_tokens(messages):
    text = tokenizer.apply_chat_template(to_parts(messages), tokenize=False)
    return len(tok(text, add_special_tokens=False).input_ids)

lengths = [n_tokens(r["messages"]) for r in train_rows]
import numpy as np
L = np.array(lengths)
print(f"tokens per example: p50 {np.percentile(L, 50):.0f}  p90 {np.percentile(L, 90):.0f}  p99 {np.percentile(L, 99):.0f}  max {L.max()}  (MAX_SEQ_LEN {MAX_SEQ_LEN})")
keep = [r for r, n in zip(train_rows, lengths) if n <= MAX_SEQ_LEN]
print(f"dropped {len(train_rows) - len(keep)} rows over MAX_SEQ_LEN")

train_dataset = [{"messages": to_parts(r["messages"])} for r in keep]
eval_dataset = [{"messages": to_parts(r["messages"])} for r in val_rows[:EVAL_ROWS]] if EVAL_ROWS else None
print(len(train_dataset), "training conversations")

## 4. Evaluation helpers: generate, score with the reward, draw

The reward is `compute_reward` from the repo: parse → render → structure score (per-stroke IoU after one scale+translation fit), 0 for a parse failure. Always look at the drawings, not just the number: the contact sheet shows the dataset's canonical strokes on the left and the model's render on the right.

In [ ]:
from IPython.display import display
from calligraphy.canvas import CanvasConfig
from calligraphy.data.mmah import get_entry
from calligraphy.splits import load_splits
from calligraphy.tools import phase1_messages
from calligraphy.score.reward import compute_reward
from make_sft import pinyin as _pinyin
from run_report import contact_sheet

CANVAS_CFG = CanvasConfig(action_frame=ACTION_FRAME)
SPLITS = load_splits()

@torch.no_grad()
def generate_batch(entries, max_new_tokens=EVAL_MAX_NEW_TOKENS, temperature=EVAL_TEMPERATURE):
    """Phase-1 answers for a list of CharacterEntry, text only (no image in the prompt)."""
    FastVisionModel.for_inference(model)
    prompts = [tokenizer.apply_chat_template(to_parts(phase1_messages(e.character, _pinyin(e), CANVAS_CFG)),
                                             add_generation_prompt=True, tokenize=False) for e in entries]
    inputs = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(max_new_tokens=max_new_tokens, use_cache=True, pad_token_id=tok.pad_token_id)
    kw.update(dict(do_sample=True, temperature=temperature, top_p=1.0) if temperature > 0 else dict(do_sample=False))
    out = model.generate(**inputs, **kw)
    return tok.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

def evaluate(split, limit, batch_size=4, tag="eval", show=True):
    chars = getattr(SPLITS, split)[:limit]
    entries = [get_entry(c) for c in chars]
    rows, t0 = [], time.time()
    for i in range(0, len(entries), batch_size):
        batch = entries[i:i + batch_size]
        for e, text in zip(batch, generate_batch(batch)):
            reward, info = compute_reward(text, e, CANVAS_CFG)
            rows.append({"index": len(rows), "character": e.character, "response": text, "reward": reward, "info": info})
        print(f"  {min(i + batch_size, len(entries))}/{len(entries)}  running mean {np.mean([r['reward'] for r in rows]):.3f}", end="\r")
    rewards = np.array([r["reward"] for r in rows])
    parsed = np.mean([r["info"]["parse_ok"] for r in rows])
    print(f"\n{tag}: {split}[:{len(rows)}]  mean reward {rewards.mean():.3f}  median {np.median(rewards):.3f}  parse rate {parsed:.2f}  "
          f"({time.time() - t0:.0f}s; untrained baseline 0.02, converter ceiling ~0.9)")
    run_dir = OUT_DIR / "evals" / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "rollouts.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    sheet = contact_sheet(rows, {r["index"]: e for r, e in zip(rows, entries)}, CANVAS_CFG, n=min(len(rows), 24))
    sheet.save(run_dir / "sheet.png")
    if show:
        display(sheet)
    return rows

### Baseline: the untrained model

Expect unreadable shapes and a reward near 0.02 (or parse failures). This is the number training has to beat.

In [ ]:
if BASELINE_CHARS:
    baseline_rows = evaluate(EVAL_SPLIT, BASELINE_CHARS, tag="baseline")

## 5. Train

Loss on the assistant turn only (`train_on_responses_only` with Qwen's chat markers), bf16 where the GPU supports it, 8-bit AdamW, cosine decay. `MAX_STEPS = 30` is a smoke test; `-1` runs `NUM_EPOCHS`.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
bf16 = torch.cuda.is_bf16_supported()

collator = UnslothVisionDataCollator(
    model, tokenizer,
    max_seq_length=MAX_SEQ_LEN,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir=str(OUT_DIR / "checkpoints"),
        run_name=RUN_NAME,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.0,
        bf16=bf16,
        fp16=not bf16,
        logging_steps=10,
        eval_strategy="steps" if eval_dataset else "no",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        seed=SEED,
        report_to=REPORT_TO,
        # required for vision-collator training (text-only rows included):
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LEN,
    ),
)

gpu = torch.cuda.get_device_properties(0)
print(f"{gpu.name}, {gpu.total_memory / 2**30:.0f} GB; {torch.cuda.max_memory_reserved() / 2**30:.1f} GB reserved before training")
steps_per_epoch = len(train_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM)
print(f"{steps_per_epoch} optimizer steps per epoch at effective batch {PER_DEVICE_BATCH * GRAD_ACCUM}")

In [ ]:
train_stats = trainer.train()
print(f"{train_stats.metrics['train_runtime'] / 60:.1f} min, final loss {train_stats.metrics.get('train_loss', float('nan')):.4f}, "
      f"peak {torch.cuda.max_memory_reserved() / 2**30:.1f} GB")
(OUT_DIR / "train_metrics.json").write_text(json.dumps({"config": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool))},
                                                        "metrics": train_stats.metrics, "log_history": trainer.state.log_history}, indent=2))

## 6. Evaluate the trained model

Same characters, same reward as the baseline. Look at the sheet first: are the strokes the right strokes in the right places? Then the numbers. The SFT target is the converter's drawing (about 0.9), so a well-trained model on `viz16`/`val` should land well above 0.5; `test_compositional` tests unseen component combinations.

In [ ]:
eval_rows = evaluate(EVAL_SPLIT, EVAL_LIMIT, tag=f"after-sft-{EVAL_SPLIT}")
worst = sorted(eval_rows, key=lambda r: r["reward"])[:3]
for r in worst:
    print(f"\n{r['character']}  reward {r['reward']:.3f}  strokes {r['info'].get('n_agent')}/{r['info'].get('n_target')}  parse_ok {r['info']['parse_ok']}")
    print(r["response"][:400])

## 7. Save

The adapter is small (about 100–300 MB). The merged 16-bit model is what vLLM or a Together custom-model deployment loads; pushing it is optional and slow.

In [ ]:
lora_dir = OUT_DIR / "lora"
model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))
print("adapter ->", lora_dir)

if HF_REPO:
    model.push_to_hub(HF_REPO, token=os.environ.get("HF_TOKEN"))
    tokenizer.push_to_hub(HF_REPO, token=os.environ.get("HF_TOKEN"))
    print("pushed adapter ->", HF_REPO)
    if PUSH_MERGED_16BIT:
        model.push_to_hub_merged(HF_REPO + "-merged", tokenizer, save_method="merged_16bit", token=os.environ.get("HF_TOKEN"))
        print("pushed merged 16-bit ->", HF_REPO + "-merged")

## 8. Together AI: managed fine-tuning and a hosted endpoint for the harness

Two uses of Together here:

- **Train the same data without a GPU.** Together's fine-tuning API takes the JSONL from section 2 as it is (OpenAI-style `messages`). `train_on_inputs=False` masks the prompt, matching the Unsloth run. The default `TOGETHER_MODEL` is the Qwen3-VL-8B id their vision guide uses; `Qwen/Qwen3.5-9B` and `Qwen/Qwen3.5-4B` are in the supported-models table (multimodal, LoRA and full). Check `tg fine-tuning model-limits <model>` if a job is rejected.
- **Serve the result for evaluation.** A fine-tuned Together model is served on a **dedicated endpoint** (serverless LoRA upload was discontinued). The endpoint is OpenAI-compatible, so the repo's harness runs the full `val` / `test_compositional` / bench evaluation against it with traces and the viewer: `scripts/run_agent.py --provider together --model <endpoint-model-string>`.

Each `fine_tuning.create` call starts a billed job; list jobs before retrying.

In [ ]:
RUN_TOGETHER_FINETUNE = False  # @param {type:"boolean"}

if RUN_TOGETHER_FINETUNE:
    assert os.environ.get("TOGETHER_API_KEY"), "set TOGETHER_API_KEY in Colab secrets"
    from together import Together
    client = Together()

    # Together wants only the messages; strip the bookkeeping keys the repo adds.
    tg_dir = OUT_DIR / "together"
    tg_dir.mkdir(exist_ok=True)
    for name, rows in (("train", train_rows), ("val", val_rows)):
        with open(tg_dir / f"{name}.jsonl", "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps({"messages": r["messages"]}, ensure_ascii=False) + "\n")

    def upload(path):
        f = client.files.upload(file=str(path), purpose="fine-tune", check=True)
        while True:
            meta = client.files.retrieve(f.id)
            if meta.processing_status == "COMPLETED":
                return f.id
            if meta.processing_status in ("INVALID_FORMAT", "FAILED"):
                raise RuntimeError(f"{path.name}: {meta.processing_status} {getattr(meta, 'validation_report', '')}")
            time.sleep(5)

    train_id, val_id = upload(tg_dir / "train.jsonl"), upload(tg_dir / "val.jsonl")
    print("uploaded", train_id, val_id)

    kwargs = dict(
        training_file=train_id,
        validation_file=val_id,
        n_evals=4,
        model=TOGETHER_MODEL,
        lora=True,
        lora_r=TOGETHER_LORA_R,
        lora_alpha=2 * TOGETHER_LORA_R,
        n_epochs=TOGETHER_EPOCHS,
        n_checkpoints=1,
        learning_rate=TOGETHER_LR,
        warmup_ratio=WARMUP_RATIO,
        train_on_inputs=False,
        suffix=TOGETHER_SUFFIX,
    )
    if "VL" in TOGETHER_MODEL or "VLM" in TOGETHER_MODEL:
        kwargs["train_vision"] = False  # phase 1 is text-only: language model only
    if os.environ.get("WANDB_API_KEY"):
        kwargs["wandb_api_key"] = os.environ["WANDB_API_KEY"]
    job = client.fine_tuning.create(**kwargs)
    (tg_dir / "job.json").write_text(json.dumps({"id": job.id, "model": TOGETHER_MODEL, "kwargs": {k: v for k, v in kwargs.items() if k != "wandb_api_key"}}, indent=2))
    print("job", job.id, "->", tg_dir / "job.json")

In [ ]:
# Poll the Together job (re-run this cell; a run takes tens of minutes to hours).
if RUN_TOGETHER_FINETUNE:
    status = client.fine_tuning.retrieve(id=job.id)
    print(status.status)
    for ev in client.fine_tuning.list_events(id=job.id).data[-5:]:
        print(" ", getattr(ev, "created_at", ""), getattr(ev, "message", ev))
    if status.status == "completed":
        print("model object:", getattr(status, "api_model_object_name", None), getattr(status, "api_model_object_id", None))

### Deploy and evaluate through the repo's harness

Once the job is `completed`, deploy the model object on a dedicated endpoint (from any shell with `TOGETHER_API_KEY` set), then run the agent harness against it. The harness records every answer, scores it with the same reward, and the viewer opens the run under **Dataset → Agent runs**.

```bash
tg endpoints deploy "<MODEL_OBJECT_NAME>" --endpoint calligraphy-sft
tg endpoints get "<ENDPOINT_ID>"            # wait for DEPLOYMENT_STATE_READY
```

```bash
uv run python scripts/run_agent.py --provider together --model "<endpoint model string>" --dataset viz16 --temperature 0 --max-tokens 4096
uv run python scripts/run_report.py <run_id> --sheet 16
```

Stop the endpoint when done (`tg endpoints rm "<ENDPOINT_ID>" --force`); a dedicated endpoint bills per hour while it is up.

The same two commands evaluate the Unsloth checkpoint if you serve the merged 16-bit model with vLLM (`--provider vllm`, section 7's `PUSH_MERGED_16BIT`).

## What comes next

- **Full evaluation of this checkpoint** on `val` (300) and `test_compositional` (500), plus the 88-row bench, through the harness; compare stroke-count accuracy and placement from `run_report.py`.
- **Phase 2 SFT**: the same converter labels replayed as `create_stroke` tool calls with one `capture_canvas` before `submit`, so the model learns the tool protocol before RL (`calligraphy.episode`).
- **GRPO** on the reward (learning rate 5e-6 to 1e-5, group size 8, temperature 1.0, drop groups with identical scores), starting from this adapter.